# Rokoko Mocap Baker Pipeline

Source files: `Assets/Scripts/Rokoko/RokokoMocapBaker.cs` (Inspector/UI layer),
`Assets/Scripts/Rokoko/RokokoJsonlBaker.cs` (bake engine), `Assets/Rokoko/Scripts/Mono/Inputs/Actor.cs`
(Rokoko package, retargeting logic reused unchanged by the baker).

Input: a `rokoko_skeleton.jsonl` recording, one JSON object per line, each line holding a
`studio_timestamp` and an `actors` array. Output: a Humanoid `AnimationClip` asset (muscle
curves + root motion curves), droppable into an Animator Controller like any other
Humanoid clip.

Every stage below runs the exact formulas and constants from the source files, in plain
Python, against small hand-written example values, so each stage's input and output is
visible as printed numbers rather than as a description of what the code does.

In [11]:
import json
import math

def q_mul(a, b):
    """Unity Quaternion multiplication, (x, y, z, w) tuples."""
    x1, y1, z1, w1 = a
    x2, y2, z2, w2 = b
    w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
    x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
    y = w1 * y2 + y1 * w2 + z1 * x2 - x1 * z2
    z = w1 * z2 + z1 * w2 + x1 * y2 - y1 * x2
    return (x, y, z, w)

def q_mul3(a, b, c):
    return q_mul(q_mul(a, b), c)

def q_conjugate(q):
    x, y, z, w = q
    return (-x, -y, -z, w)

def q_inverse(q):
    x, y, z, w = q
    norm_sq = x * x + y * y + z * z + w * w
    cx, cy, cz, cw = q_conjugate(q)
    return (cx / norm_sq, cy / norm_sq, cz / norm_sq, cw / norm_sq)

def q_normalize(q):
    x, y, z, w = q
    n = math.sqrt(x * x + y * y + z * z + w * w)
    return (x / n, y / n, z / n, w / n)

def cross(a, b):
    return (
        a[1] * b[2] - a[2] * b[1],
        a[2] * b[0] - a[0] * b[2],
        a[0] * b[1] - a[1] * b[0],
    )

def rotate_vector(q, v):
    """Rotates vector v by quaternion q (Unity's `q * v`)."""
    qn = q_normalize(q)
    vq = (v[0], v[1], v[2], 0.0)
    rx, ry, rz, _ = q_mul3(qn, vq, q_conjugate(qn))
    return (rx, ry, rz)

def fmt(q, nd=4):
    return tuple(round(c, nd) for c in q)

## Stage 0a — T-pose capture

`Actor.CalculateTPose()` (`Actor.cs:98-105`) runs when the Inspector's "Assign T-Pose Now"
button is clicked, with the character posed in T-pose in the Scene view at that instant.
`InitializeCharacterTPose()` (`Actor.cs:116-128`) walks every `HumanBodyBones` value and
records `boneTransform.rotation` — the bone's *current world rotation, right now* — into
`characterTPose`. Nothing here derives from the mocap data; it is a live snapshot of
whatever pose the character's Transforms happen to be in at the moment of the click.

The values below stand in for a snapshot taken on some example character.

In [2]:
character_t_pose = {
    "Hips":          (0.0, 0.0, 0.0, 1.0),
    "Spine":         (0.0, 0.0, 1.0, 0.0),
    "Chest":         (0.0, 0.0, 1.0, 0.0),
    "LeftUpperArm":  (-0.5, -0.5, 0.5, -0.5),
    "RightUpperArm": (0.5, -0.5, 0.5, 0.5),
    "LeftUpperLeg":  (0.0, 0.707, 0.0, 0.707),
    "RightUpperLeg": (0.0, -0.707, 0.0, 0.707),
}

for bone, q in character_t_pose.items():
    print(f"{bone:<15} {fmt(q)}")

Hips            (0.0, 0.0, 0.0, 1.0)
Spine           (0.0, 0.0, 1.0, 0.0)
Chest           (0.0, 0.0, 1.0, 0.0)
LeftUpperArm    (-0.5, -0.5, 0.5, -0.5)
RightUpperArm   (0.5, -0.5, 0.5, 0.5)
LeftUpperLeg    (0.0, 0.707, 0.0, 0.707)
RightUpperLeg   (0.0, -0.707, 0.0, 0.707)


## Stage 0b — rotation offsets

`InitializeBoneOffsets()` (`Actor.cs:133-137`) calls `CalculateRotationOffsets()`
(`Actor.cs:323-334`), which for every bone present in `characterTPose` computes:

```
offsets[bone] = Inverse(SmartsuitTPose[bone]) * characterTPose[bone]
```

`SmartsuitTPose` (`Actor.cs:339-395`) is a fixed dictionary hardcoded in the Rokoko
package — the Smartsuit's own reference T-pose rotation per bone, in the suit's world
convention. It is the same for every character; only `characterTPose` changes. A subset
of its real values (verbatim from `Actor.cs`):

In [3]:
smartsuit_t_pose = {
    "Hips":          (0.0, 0.0, 0.0, 1.0),
    "Spine":         (0.0, 0.0, 1.0, 0.0),
    "Chest":         (0.0, 0.0, 1.0, 0.0),
    "LeftUpperArm":  (-0.5, -0.5, 0.5, -0.5),
    "RightUpperArm": (0.5, -0.5, 0.5, 0.5),
    "LeftUpperLeg":  (0.0, 0.707, 0.0, 0.707),
    "RightUpperLeg": (0.0, -0.707, 0.0, 0.707),
}

offsets = {}
for bone in smartsuit_t_pose:
    offsets[bone] = q_mul(q_inverse(smartsuit_t_pose[bone]), character_t_pose[bone])
    print(f"{bone:<15} offset = {fmt(offsets[bone])}")

Hips            offset = (0.0, 0.0, 0.0, 1.0)
Spine           offset = (0.0, 0.0, 0.0, 1.0)
Chest           offset = (0.0, 0.0, 0.0, 1.0)
LeftUpperArm    offset = (0.0, 0.0, 0.0, 1.0)
RightUpperArm   offset = (0.0, 0.0, 0.0, 1.0)
LeftUpperLeg    offset = (0.0, 0.0, 0.0, 1.0)
RightUpperLeg   offset = (0.0, 0.0, 0.0, 1.0)


In this example every bone's `characterTPose` value matches `SmartsuitTPose` exactly, so
every offset comes out as the identity quaternion `(0, 0, 0, 1)` — expected, since this
example character's rest pose was defined identically to the suit's. A character rigged
with a rotated rest pose on some bone (a twisted forearm roll, a different shoulder
zero-pose) would show a non-identity offset there, and only there.

## Stage 1 — reading one JSONL line

Each line of `rokoko_skeleton.jsonl` deserializes into `RawLine` (`RokokoJsonlBaker.cs:33-38`):
`studio_timestamp` plus an `actors` array of `RawActorEntry` (`RokokoJsonlBaker.cs:25-31`):
`name`, `hip_height`, and `bones` (a `BodyFrame`, `JsonLiveSerializerV3.cs:83-161` — one
`ActorJointFrame { position, rotation }` per named joint). A trimmed example line, two
joints only (real recordings carry roughly fifty):

In [4]:
raw_line_text = '''
{"studio_timestamp": 12.033,
 "actors": [
   {"name": "DemoProfile", "hip_height": 0.92,
    "bones": {
      "hip":          {"position": {"x": 0.01, "y": 0.92, "z": -0.02}, "rotation": {"x": 0.0,  "y": 0.05, "z": 0.0,  "w": 0.999}},
      "leftUpperArm": {"position": {"x": 0.2,  "y": 1.4,  "z": 0.0},  "rotation": {"x": -0.46, "y": -0.54, "z": 0.46, "w": -0.54}}
    }}
 ]}
'''

raw_line = json.loads(raw_line_text)
print(json.dumps(raw_line, indent=2))

{
  "studio_timestamp": 12.033,
  "actors": [
    {
      "name": "DemoProfile",
      "hip_height": 0.92,
      "bones": {
        "hip": {
          "position": {
            "x": 0.01,
            "y": 0.92,
            "z": -0.02
          },
          "rotation": {
            "x": 0.0,
            "y": 0.05,
            "z": 0.0,
            "w": 0.999
          }
        },
        "leftUpperArm": {
          "position": {
            "x": 0.2,
            "y": 1.4,
            "z": 0.0
          },
          "rotation": {
            "x": -0.46,
            "y": -0.54,
            "z": 0.46,
            "w": -0.54
          }
        }
      }
    }
  ]
}


## Stage 2 — actor selection

`BakeToClip` (`RokokoJsonlBaker.cs:146-156`) scans `raw.actors` for an entry whose `name`
matches `actor.profileName`, falling back to `actors[0]` if none matches — a mocap take
recorded with multiple performers only bakes the one this character is assigned to.

In [5]:
def select_actor_entry(raw, profile_name):
    for a in raw["actors"]:
        if a["name"] == profile_name:
            return a
    return raw["actors"][0]

src = select_actor_entry(raw_line, profile_name="DemoProfile")
print(json.dumps(src, indent=2))

{
  "name": "DemoProfile",
  "hip_height": 0.92,
  "bones": {
    "hip": {
      "position": {
        "x": 0.01,
        "y": 0.92,
        "z": -0.02
      },
      "rotation": {
        "x": 0.0,
        "y": 0.05,
        "z": 0.0,
        "w": 0.999
      }
    },
    "leftUpperArm": {
      "position": {
        "x": 0.2,
        "y": 1.4,
        "z": 0.0
      },
      "rotation": {
        "x": -0.46,
        "y": -0.54,
        "z": 0.46,
        "w": -0.54
      }
    }
  }
}


## Stage 3 — building the `ActorFrame`

`RokokoJsonlBaker.cs:158-164` repackages the raw entry into Rokoko's own `ActorFrame`
struct: `name`, `meta.hasBody = true` (unconditionally — this baker only ever bakes body
data, not gloves/face), `dimensions.hipHeight = hip_height`, `body = bones`.

In [6]:
actor_frame = {
    "name": src["name"],
    "meta": {"hasBody": True},
    "dimensions": {"hipHeight": src["hip_height"]},
    "body": src["bones"],
}
print(json.dumps(actor_frame, indent=2))

{
  "name": "DemoProfile",
  "meta": {
    "hasBody": true
  },
  "dimensions": {
    "hipHeight": 0.92
  },
  "body": {
    "hip": {
      "position": {
        "x": 0.01,
        "y": 0.92,
        "z": -0.02
      },
      "rotation": {
        "x": 0.0,
        "y": 0.05,
        "z": 0.0,
        "w": 0.999
      }
    },
    "leftUpperArm": {
      "position": {
        "x": 0.2,
        "y": 1.4,
        "z": 0.0
      },
      "rotation": {
        "x": -0.46,
        "y": -0.54,
        "z": 0.46,
        "w": -0.54
      }
    }
  }
}


## Stage 4 — `Actor.UpdateActor` → `UpdateSkeleton` → `UpdateBone`

`UpdateActor` (`Actor.cs:162-177`) sets `profileName = actorFrame.name`, then — since
`hasBody` is true — calls `UpdateSkeleton` (`Actor.cs:249-269`), which walks every
`HumanBodyBones` value, pulls the matching `ActorJointFrame` out of `body` by name, and
calls `UpdateBone` (`Actor.cs:274-316`) for each one.

Only `HumanBodyBones.Hips` gets `shouldUpdatePosition = true` — every other bone is
rotation-only, since a Humanoid rig derives limb position from parent rotations via
forward kinematics; the hips are the one bone that needs an explicit world position.

`UpdateBone`'s rotation branch, for this project's default `rotationSpace = Offset`
(`Actor.cs:312-315`):

```
boneTransform.rotation = Hips.parent.rotation * worldRotation * offsets[bone]
```

`worldRotation` is this frame's raw rotation for the bone, straight out of the JSONL, in
the Smartsuit's own convention. `offsets[bone]` (Stage 0b) re-expresses that rotation in
this specific character's rest-pose convention. `Hips.parent.rotation` re-anchors the
result to wherever this character's root sits in the scene. This one line is the actual
per-bone retargeting happening on every frame of every bake.

In [7]:
def update_bone_offset_mode(hips_parent_rotation, world_rotation, offset):
    return q_mul3(hips_parent_rotation, world_rotation, offset)

hips_parent_rotation = (0.0, 0.0, 0.0, 1.0)  # character root not rotated in this example

left_upper_arm_raw = (-0.46, -0.54, 0.46, -0.54)  # straight from Stage 1's JSONL line

retargeted_left_upper_arm = update_bone_offset_mode(
    hips_parent_rotation,
    left_upper_arm_raw,
    offsets["LeftUpperArm"],
)
print("LeftUpperArm final local rotation:", fmt(retargeted_left_upper_arm))

LeftUpperArm final local rotation: (-0.46, -0.54, 0.46, -0.54)


## Stage 4b — hip position

`UpdateBone`'s position branch (`Actor.cs:287-298`), for this project's default
`positionSpace = Self`:

```
boneTransform.position = parent.rotation * worldPosition + parent.position
```

and, only if `adjustHipHeightBasedOnStudioActor` is enabled (`Actor.cs:262-264`, off by
default in this project):

```
worldPosition.y -= (actorFrame.dimensions.hipHeight - characterHipHeight)
```

which re-levels a mocap performer's hip height difference against this specific
character's own leg length before the position conversion above runs.

In [8]:
def self_space_position(parent_rotation, parent_position, world_position):
    rx, ry, rz = rotate_vector(parent_rotation, world_position)
    px, py, pz = parent_position
    return (rx + px, ry + py, rz + pz)

hip_world_position_raw = (0.01, 0.92, -0.02)   # from Stage 1's JSONL line
hips_parent_position = (0.0, 0.0, 0.0)          # character root at scene origin

character_hip_height = 0.88   # this character's own resting hip height
actor_hip_height = actor_frame["dimensions"]["hipHeight"]  # 0.92, from Stage 3

adjust_hip_height = False  # project default
if adjust_hip_height:
    hx, hy, hz = hip_world_position_raw
    hip_world_position_raw = (hx, hy - (actor_hip_height - character_hip_height), hz)

final_hip_position = self_space_position(hips_parent_rotation, hips_parent_position, hip_world_position_raw)
print("Hips final local position:", tuple(round(c, 4) for c in final_hip_position))

Hips final local position: (0.01, 0.92, -0.02)


## Stage 5 — sampling the posed rig

After `UpdateActor` finishes mutating every bone Transform for this frame,
`RokokoJsonlBaker.cs:173` calls `poseHandler.GetHumanPose(ref pose)`. `HumanPoseHandler`
is Unity's own Mecanim component; it reads the now-posed Transform hierarchy and converts
it into `HumanPose` — roughly 95 normalized `muscles[]` values (one per Humanoid degree
of freedom, in the `[-1, 1]` range defined by this specific avatar's configured muscle
limits) plus `bodyPosition`/`bodyRotation` for the root. This conversion runs inside the
Unity engine against this avatar's imported muscle-limit data and is not something to
reproduce outside Unity; everything on either side of it is plain data transformation, as
shown in every other stage here.

## Stage 6 — timestamp to keyframe time

`RokokoJsonlBaker.cs:168-171`, run once per JSONL line, in this exact order:

```
dt = 1/30                         if lastTs < 0
     max(studio_timestamp - lastTs, 1/240)   otherwise
if applied > 0: time += dt
lastTs = studio_timestamp
```

The first applied frame always lands at `time = 0` regardless of its own `dt` (that `dt`
only affects the *next* frame's advance); the `1/240` floor guards against a near-zero or
negative timestamp delta producing a degenerate or reversed keyframe.

In [9]:
def accumulate_times(timestamps):
    last_ts = -1.0
    time = 0.0
    applied = 0
    result = []
    for ts in timestamps:
        dt = (1 / 30) if last_ts < 0 else max(ts - last_ts, 1 / 240)
        if applied > 0:
            time += dt
        last_ts = ts
        applied += 1
        result.append(round(time, 5))
    return result

example_timestamps = [12.000, 12.033, 12.066, 12.100, 12.250]
print(accumulate_times(example_timestamps))

[0.0, 0.033, 0.066, 0.1, 0.25]


## Stage 7 — writing the `AnimationClip`

For every applied frame, `RokokoJsonlBaker.cs:175-184` pushes one `Keyframe(time, value)`
per channel: `HumanTrait.MuscleCount` muscle channels (named by Unity's own
`HumanTrait.MuscleName[m]`, e.g. `"Spine Front-Back"`, `"Left Arm Twist In-Out"`) plus
three `RootT.x/y/z` and four `RootQ.x/y/z/w` channels for root motion.

`RokokoJsonlBaker.cs:203-213` then calls, once per channel:

```
clip.SetCurve("", typeof(Animator), channelName, new AnimationCurve(keyframes))
```

Targeting `typeof(Animator)` with these specific channel names — rather than recording
raw Transform position/rotation curves per bone, the way a `GameObjectRecorder` would —
is what makes the resulting `AnimationClip` a genuine Humanoid muscle clip: retargetable
to any other Humanoid avatar, and usable in an Animator Controller alongside other
Humanoid clips.

In [10]:
muscle_keyframes_example = {
    "Spine Front-Back": [(0.0, 0.02), (0.033, 0.021), (0.066, 0.019)],
    "Left Arm Twist In-Out": [(0.0, -0.10), (0.033, -0.11), (0.066, -0.09)],
}
root_t_keyframes_example = {
    "RootT.x": [(0.0, 0.01), (0.033, 0.012)],
    "RootT.y": [(0.0, 0.92), (0.033, 0.921)],
    "RootT.z": [(0.0, -0.02), (0.033, -0.019)],
}
root_q_keyframes_example = {
    "RootQ.x": [(0.0, 0.0), (0.033, 0.001)],
    "RootQ.y": [(0.0, 0.05), (0.033, 0.052)],
    "RootQ.z": [(0.0, 0.0), (0.033, 0.0)],
    "RootQ.w": [(0.0, 0.999), (0.033, 0.999)],
}

for channel, keys in {**muscle_keyframes_example, **root_t_keyframes_example, **root_q_keyframes_example}.items():
    print(f"{channel:<24} {keys}")

Spine Front-Back         [(0.0, 0.02), (0.033, 0.021), (0.066, 0.019)]
Left Arm Twist In-Out    [(0.0, -0.1), (0.033, -0.11), (0.066, -0.09)]
RootT.x                  [(0.0, 0.01), (0.033, 0.012)]
RootT.y                  [(0.0, 0.92), (0.033, 0.921)]
RootT.z                  [(0.0, -0.02), (0.033, -0.019)]
RootQ.x                  [(0.0, 0.0), (0.033, 0.001)]
RootQ.y                  [(0.0, 0.05), (0.033, 0.052)]
RootQ.z                  [(0.0, 0.0), (0.033, 0.0)]
RootQ.w                  [(0.0, 0.999), (0.033, 0.999)]


## Rig safety around the whole loop

`RokokoJsonlBaker.cs:119-126` snapshots every Transform's `localPosition`/`localRotation`
under `actor.animator.transform` before the frame loop starts (`GetComponentsInChildren
<Transform>`), because `UpdateActor` mutates the live scene rig frame by frame to produce
each pose to sample. `RokokoJsonlBaker.cs:189-198` restores every snapshot in a `finally`
block — on normal completion or on an exception mid-loop — so baking a clip never leaves
the character's Transforms displaced in the scene.

`RokokoJsonlBaker.cs:96-97` also invokes `Actor.InitializeAnimatorHumanBones` and
`Actor.InitializeBoneOffsets` through reflection before the loop starts. Both are normally
called from `Actor.Awake()` (`Actor.cs:76-77`), which only runs in Play Mode; baking runs
entirely in the Editor without entering Play Mode, so without this call `offsets` (Stage
0b) and the cached bone-transform dictionary would still be empty at bake time.

## Inspector layer — `RokokoMocapBaker` / `RokokoMocapBakerEditor`

`RokokoMocapBaker` (`RokokoMocapBaker.cs:20-27`) itself holds only two fields:
`jsonlPath` and `outputFolder`. Every action lives in its `CustomEditor`:

- `EnsureActorSetUp` (`RokokoMocapBaker.cs:126-152`), run on every Inspector repaint: adds
  an `Actor` component if the character doesn't have one yet, wires `actor.animator`, and
  calls `CalculateTPose()` exactly once — only while `characterTPose` is still empty, so a
  T-pose already captured is never silently overwritten by whatever pose the character
  happens to be in during a later repaint.
- The "Assign T-Pose Now" / "Recalculate T-Pose" button (`RokokoMocapBaker.cs:68-76`) calls
  `actor.CalculateTPose()` on demand, then checks `actor.isValidTpose` — `Actor.cs:204-228`
  rejects the capture unless the hand-to-hand direction is within `0.99` dot product of
  world right and the spine-to-chest direction is within `0.99` dot product of world up.
- The drop area / Browse button (`RokokoMocapBaker.cs:154-179`, `81-89`) sets `jsonlPath`.
- `RokokoJsonlBaker.Validate` (`RokokoJsonlBaker.cs:44-53`) gates the "Bake Animation Clip"
  button: an `Actor` must exist, its `Animator` must be assigned and Humanoid, and
  `characterTPose` must be non-empty.
- `Bake` (`RokokoMocapBaker.cs:181-199`) normalizes `outputFolder` to a forward-slash,
  in-`Assets`-rooted path, creates it if missing, builds a unique output filename as
  `{characterName}_{jsonlBaseName}.anim` via `AssetDatabase.GenerateUniqueAssetPath`, and
  calls `RokokoJsonlBaker.RunBake`, which wraps `BakeToClip` (everything in Stages 0–7
  above) with a progress bar and a completion dialog reporting the frame count.